In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_vggnet16
from train import WarmUpCosine, CustomWeightDecaySGD, LastNSaver, make_train_ds, make_test_ds

In [4]:
num_classes = 3
initial_lr = 0.1
weight_decay = 1e-4
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
NN_16=[32,32,64,64,128,128,256,256]

In [7]:
model = build_vggnet16(NN_16,num_class=4)

In [8]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = CustomWeightDecaySGD(
    weight_decay=weight_decay,
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=True
)

In [9]:
loss_fn=tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [10]:
saver = LastNSaver(n=20)

In [11]:
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.2   # 0.2~0.4 都常用；过拟合明显就加大
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 16s - loss: 1.1949 - accuracy: 0.4862 - val_loss: 1.3041 - val_accuracy: 0.3228 - 16s/epoch - 105ms/step
Epoch 2/200
156/156 - 2s - loss: 0.9365 - accuracy: 0.6523 - val_loss: 0.7463 - val_accuracy: 0.7122 - 2s/epoch - 15ms/step
Epoch 3/200
156/156 - 2s - loss: 0.8034 - accuracy: 0.7298 - val_loss: 1.1615 - val_accuracy: 0.6310 - 2s/epoch - 15ms/step
Epoch 4/200
156/156 - 2s - loss: 0.7318 - accuracy: 0.7657 - val_loss: 1.8687 - val_accuracy: 0.5487 - 2s/epoch - 15ms/step
Epoch 5/200
156/156 - 2s - loss: 0.6933 - accuracy: 0.7856 - val_loss: 1.8704 - val_accuracy: 0.5505 - 2s/epoch - 15ms/step
Epoch 6/200
156/156 - 2s - loss: 0.6488 - accuracy: 0.8119 - val_loss: 0.5639 - val_accuracy: 0.8008 - 2s/epoch - 15ms/step
Epoch 7/200
156/156 - 2s - loss: 0.6199 - accuracy: 0.8232 - val_loss: 0.6074 - val_accuracy: 0.7720 - 2s/epoch - 15ms/step
Epoch 8/200
156/156 - 2s - loss: 0.5981 - accuracy: 0.8342 - val_loss: 0.8879 - val_accuracy: 0.7038 - 2s/epoch - 15ms/step
Epoch

In [12]:
model.save("VGG_11.h5")